<a href="https://colab.research.google.com/github/ArjunHirani/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane



## 1. My lane (or freestyle) and why

**Freestyle — Growth / Recovery / Momentum Prediction.**

Instead of scoring pages on their current-window state (the four predefined lanes), I want to predict *future* movement: given a page's signals over a prior window, will it decline, hold, or gain momentum over the next window? This is deliberately harder than the predefined lanes — it requires building my own time-aware labels from the full warehouse (`fact_content_daily_performance`, ~79M rows) rather than using a same-window proxy label, and it demands strict leakage control plus time-aware/client-grouped validation rather than a simple random split.

I'm choosing this because the starter pipeline I already ran (Notebooks 01-02) uses a same-window proxy label (`is_declining_label = trend_direction == "down"`) — useful for learning the workflow, but not a future prediction. A future-window label (prior 90 days -> next 30 days outcome) is a materially harder and more honest prediction problem, and the starter data itself already hints this is worth pursuing (see Section 3).

In [1]:
# Supporting note for Section 1: the starter CSV already ships paired prior/current
# 30-day windows for several metrics. That's a strong hint this lane is a natural,
# already-scaffolded extension of the existing schema -- not an invented problem.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

window_pair_cols = [c for c in df.columns if "last_30d" in c or "prev_30d" in c]
print("Existing prior/current window column pairs already in the starter schema:")
print(window_pair_cols)

Existing prior/current window column pairs already in the starter schema:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 2. The question: decision, action, cost of a wrong call

**Decision:** Given a page's search and engagement signals from the prior 90 days, will it decline, recover, or gain momentum over the next 30 days? Which pages should a content reviewer look at *before* the decline is obvious, rather than after?

**Who acts, and how:** A content strategist or SEO reviewer with limited weekly review capacity (realistically ~20-50 pages a week, not the whole site). They use the ranked future-risk list to decide which pages to refresh, protect, or monitor *this week*, prioritizing pages the model flags as high-risk before the decline shows up in a standard monthly report.

**Cost of a wrong call:**
- False positive (flagged as at-risk, but it wasn't): wastes a reviewer's limited hours on a page that didn't need attention — the opportunity cost is another page that *did* need it going unreviewed.
- False negative (page was heading for decline, model missed it): the page loses visibility silently for weeks before anyone notices in a standard report, by which point recovery is harder and slower.
- Given limited review capacity, **precision@K** (of the top K flagged pages, how many actually moved) matters more than overall accuracy — a reviewer who checks the top 20-50 pages needs those to be right, not the bottom thousands.

**Why data/ML helps here, not just a rule:** A simple rule like "flag pages with low `avg_position` and falling impressions" can't see *combinations* of weak signals (position, CTR, engagement, freshness, content type) shifting together over time, and it can't be validated on whether it predicts the *future* — only whether it matches the *current* state. A model trained on prior-window features against a true future-window outcome, validated with time-aware and client-grouped splits, can surface that combination — and critically, we can measure whether it's actually better than the naive rule, rather than assuming it.

In [2]:
# Supporting number for Section 2: quantify the reviewer-capacity problem this lane addresses.
declining_count = df["trend_direction"].str.lower().eq("down").sum()
total = len(df)
reviewer_capacity_per_week = 50

print(f"Declining pages in this snapshot: {declining_count} of {total} ({declining_count/total:.1%})")
print(f"At {reviewer_capacity_per_week} pages/week reviewer capacity, "
      f"weeks to review ALL currently-declining pages: {declining_count/reviewer_capacity_per_week:.0f}")
print(f"That's roughly {declining_count/reviewer_capacity_per_week/52:.1f} years -- for just ONE 90-day snapshot.")
print("This is why RANKING (who first) matters as much as DETECTION (who at all).")

Declining pages in this snapshot: 16262 of 30000 (54.2%)
At 50 pages/week reviewer capacity, weeks to review ALL currently-declining pages: 325
That's roughly 6.3 years -- for just ONE 90-day snapshot.
This is why RANKING (who first) matters as much as DETECTION (who at all).


## 3. Quick look at the data (2-3 real numbers)


In [3]:
# Number 1: how much of the inventory is already flagged as declining under the CURRENT-window proxy
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"1) Share of pages currently 'declining' (90-day proxy label): {declining_rate:.3f}")

# Number 2: how many distinct clients exist -- sets the scale for client-grouped time-aware validation
n_clients = df["client_id"].nunique()
print(f"2) Distinct clients in starter slice: {n_clients}")

# Number 3 & 4: does the most recent 30-day window already show momentum that DISAGREES
# with the broader 90-day trend label -- i.e. is there real, non-trivial short-term
# signal here that a same-window snapshot proxy would miss?
df["click_momentum"] = df["clicks_last_30d"] - df["clicks_prev_30d"]

opposite_short_term = (
    ((df["trend_direction"].str.lower() == "down") & (df["click_momentum"] > 0)) |
    ((df["trend_direction"].str.lower() == "up") & (df["click_momentum"] < 0))
)
print(f"3) Share of pages where last-30d click momentum CONTRADICTS the 90-day trend label: {opposite_short_term.mean():.3f}")

corr = df["click_momentum"].corr(df["trend_pct"])
print(f"4) Correlation between last-30d click momentum and 90-day trend_pct: {corr:.3f}")

1) Share of pages currently 'declining' (90-day proxy label): 0.542
2) Distinct clients in starter slice: 32
3) Share of pages where last-30d click momentum CONTRADICTS the 90-day trend label: 0.110
4) Correlation between last-30d click momentum and 90-day trend_pct: 0.025


**Reading these numbers:**

- At **54.2%** declining under the current proxy label, and **32 clients**, there's both enough signal and enough client diversity for a proper client-grouped, time-aware validation split later.
- The standout number is **11.0%**: roughly 1 in 9 pages already shows last-30-day click momentum that *contradicts* its own 90-day trend label. That's not noise-level — it's a meaningful chunk of the inventory where a same-window snapshot proxy would give a reviewer the wrong current read, let alone a future one.
- The near-zero correlation (**0.025**) between that short-term momentum and the 90-day `trend_pct` says short-term movement isn't a simple linear echo of the longer trend. That's exactly the "real signal, but tangled and shifting" situation the framing guide says is where ML earns its place over a hand rule — a rule-writer would have to guess how to combine short-term momentum with everything else, and couldn't tell if the guess was right without the kind of held-out, future-window test this lane requires.

## 4. Careful words: what I can and can't claim


**What I can claim:**
- **Observed / directional patterns**: e.g. "pages with X combination of prior-window signals were more likely to decline over the following 30 days, in this dataset, in this validation split."
- **Decision-support, not decision-making**: a ranked list a reviewer can use to prioritize limited attention — never an automatic action taken on their behalf.
- **Relative, measured performance**: precision@K compared honestly against a fixed baseline rule, on a held-out set that never shares a client with training.

**What I will never claim:**
- Causal proof that any specific factor *causes* decline or recovery — only that it's associated with it in this data.
- Any claim about how a search engine's ranking algorithm works — this is about a client's own content performance signals, not reverse-engineering anyone's algorithm.
- That a same-window proxy label result generalizes to true future prediction until I've actually built and validated the future-window label myself — until then, this remains a provisional, stated plan, not a result.
- 100% accuracy or a guarantee for any individual page — only aggregate, validated tendencies across the ranked list.

In [4]:
# Supporting check for Section 4: make explicit which columns must NEVER be used as
# features for this lane, per the leakage lesson from Notebook 02 and the data skill's
# label-trap warning. Writing this down now, before any modeling, is the point.
never_features = ["trend_direction", "trend_pct"]
pseudonym_ids_not_features = ["content_id", "client_id"]

print("Columns that must NEVER be used as model features (label leakage):", never_features)
print("Columns that are identifiers only, never features (may be used for grouping/splitting):",
      pseudonym_ids_not_features)
print()
print("Reminder for the eventual future-window label: it must be built from data STRICTLY")
print("after the feature window ends, using the warehouse's report_date, not from any")
print("column (like trend_direction/trend_pct) that summarizes the SAME window as the features.")

Columns that must NEVER be used as model features (label leakage): ['trend_direction', 'trend_pct']
Columns that are identifiers only, never features (may be used for grouping/splitting): ['content_id', 'client_id']

Reminder for the eventual future-window label: it must be built from data STRICTLY
after the feature window ends, using the warehouse's report_date, not from any
column (like trend_direction/trend_pct) that summarizes the SAME window as the features.
